# WeaveGuard AI — Étape 08 : fusion YOLO + PatchCore (évaluation et format de sortie)
**Règle de fusion (cahier des charges §7.4), sans paramètre à régler :**
- toutes les détections YOLO (confiance >= seuil calibré) sont conservées, avec leur **classe et leur score** ;
- une région PatchCore qui recouvre une détection YOLO est **rattachée** comme *corroboration* (rien n'est modifié) ;
- une région PatchCore sans équivalent YOLO devient une région distincte : **« Anomalie suspectée — score de distance »**, sans classe.

**Mesures** : rappel de YOLO seul / PatchCore seul / union ; défauts manqués par YOLO et retrouvés par PatchCore (par classe) ; régions supplémentaires à revoir par image ; corroboration des bonnes détections vs des fausses.
Protocole : validation d'abord, **test une seule fois** (seuils déjà calibrés, rien n'est réglé ici). Si `data/external_test/` existe, comptage des alertes sur des tissus inconnus.

In [ ]:
# 1) Drive + dépendances + GPU
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics
import torch
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN')
assert torch.cuda.is_available(), 'Active un GPU (T4)'

In [ ]:
# 2) Configuration : modèles exportés (YOLO, PatchCore) et dataset v2
from pathlib import Path
import json, hashlib, zipfile, random, collections, datetime, time, math
import numpy as np, pandas as pd
from PIL import Image, ImageOps, ImageDraw
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
from scipy import ndimage
from tqdm.auto import tqdm
from ultralytics import YOLO

WG = Path('/content/drive/MyDrive/WeaveGuard'); DRIVE_OUT = WG / 'datasets'
V2 = Path('/content/weaveguard_dataset_v2'); SEED = 42; dev = torch.device('cuda')
YOLO_DIR = WG / 'models' / 'weaveguard_yolo_v1'; PC_DIR = WG / 'models' / 'patchcore' / 'patchcore_v1'
OUT = WG / 'models' / 'fusion' / 'fusion_v1'; OUT.mkdir(parents=True, exist_ok=True)
EXT_DIR = WG / 'data' / 'external_test'; EXT_LONG_SIDE = 1448     # taille longue des photos externes (échelle du dataset)

ycard = json.load(open(YOLO_DIR / 'model_card.json')); CONF, IMGSZ = ycard['conf_threshold']['value'], ycard['imgsz']
calib = json.load(open(PC_DIR / 'calibration.json')); TAU, MIN_AREA = calib['tau_global'], calib['min_region_area']
bank_pkg = torch.load(PC_DIR / 'memory_bank.pt', map_location='cpu', weights_only=False)
PCFG = bank_pkg['config']; BANK = bank_pkg['bank'].float().to(dev)
SCALE, BLUR_SIGMA, FEAT_DIM, BACKBONE = PCFG['scale'], PCFG['blur_sigma'], PCFG['feature_dim'], PCFG['backbone']
print(f"YOLO {ycard['selected_experiment']} conf={CONF:.3f} imgsz={IMGSZ} | PatchCore tau={TAU:.3f} scale={SCALE} bank={tuple(BANK.shape)}")

if not V2.exists(): zipfile.ZipFile(DRIVE_OUT / 'weaveguard_dataset_v2.zip').extractall(V2)
stats = json.load(open(DRIVE_OUT / 'dataset_stats_v2.json')); man = pd.read_csv(V2 / 'manifest.csv')
h = hashlib.sha256('\n'.join(sorted(f'{a}|{b}|{c}' for a, b, c in zip(man.name, man.md5, man.split))).encode()).hexdigest()
assert h == stats['dataset_hash_sha256'], 'dataset local != v2 de référence'
NAMES = stats['classes']

In [ ]:
# 3) PatchCore : même extracteur et même prétraitement que le notebook 05
class Extractor(nn.Module):
    def __init__(self):
        super().__init__(); m = getattr(torchvision.models, BACKBONE)(weights='IMAGENET1K_V1')
        self.stem = nn.Sequential(m.conv1, m.bn1, m.relu, m.maxpool); self.l1, self.l2, self.l3 = m.layer1, m.layer2, m.layer3
        for p in self.parameters(): p.requires_grad = False
    @torch.no_grad()
    def forward(self, x):
        f2 = self.l2(self.l1(self.stem(x))); f3 = self.l3(f2)
        f2 = F.avg_pool2d(f2, 3, 1, 1); f3 = F.avg_pool2d(f3, 3, 1, 1)
        f3 = F.interpolate(f3, size=f2.shape[-2:], mode='bilinear', align_corners=False)
        f = torch.cat([f2, f3], 1); B, C, hh, ww = f.shape
        f = f.permute(0, 2, 3, 1).reshape(B * hh * ww, C)
        return F.adaptive_avg_pool1d(f.unsqueeze(1), FEAT_DIM).squeeze(1).reshape(B, hh, ww, FEAT_DIM)
extractor = Extractor().to(dev).eval()
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
def to_tensor(im):
    W, H = im.size; im = im.resize((round(W * SCALE), round(H * SCALE)), Image.BILINEAR)
    return (torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.).permute(2, 0, 1) - MEAN) / STD
@torch.no_grad()
def nn_dist(Q, bank, chunk=4096):
    bn = (bank ** 2).sum(1); out = torch.empty(Q.shape[0], device=dev)
    for i in range(0, Q.shape[0], chunk):
        q = Q[i:i + chunk].float(); d2 = (q ** 2).sum(1, keepdim=True) - 2 * q @ bank.T + bn[None]
        out[i:i + chunk] = d2.min(1).values.clamp_min(0).sqrt()
    return out
@torch.no_grad()
def score_map(x):
    f = extractor(x[None].to(dev)); _, hh, ww, d = f.shape
    dist = nn_dist(f.reshape(-1, d), BANK).reshape(1, 1, hh, ww)
    m = F.interpolate(dist, size=tuple(x.shape[-2:]), mode='bilinear', align_corners=False)[0, 0].cpu().numpy()
    return ndimage.gaussian_filter(m, sigma=BLUR_SIGMA)
model = YOLO(str(YOLO_DIR / 'best.pt'))

In [ ]:
# 4) Fusion + mesures (sans PyTorch)
import numpy as np, collections
from scipy import ndimage

def regions_from_map(m, tau, min_area):
    lab, _ = ndimage.label(m >= tau); regs = []
    for k, sl in enumerate(ndimage.find_objects(lab), 1):
        if sl is None: continue
        mask = lab[sl] == k; area = int(mask.sum())
        if area < min_area: continue
        v = m[sl][mask]; regs.append(dict(x0=sl[1].start, y0=sl[0].start, x1=sl[1].stop, y1=sl[0].stop, area_px=area, score_max=float(v.max()), score_mean=float(v.mean())))
    return regs
def rescale(regs, f): return [dict(r, x0=r['x0'] * f, y0=r['y0'] * f, x1=r['x1'] * f, y1=r['y1'] * f) for r in regs]
def inter(a, b):
    w = min(a[2], b[2]) - max(a[0], b[0]); h = min(a[3], b[3]) - max(a[1], b[1]); return max(0., w) * max(0., h)
def area(b): return max(1e-9, (b[2] - b[0]) * (b[3] - b[1]))
def iou(a, b):
    i = inter(a, b); return i / (area(a) + area(b) - i)
def overlaps(box, reg):
    # recouvrement région/boîte : >= 10 % de la boîte OU >= 50 % de la région (même règle que le notebook 05)
    rb = (reg['x0'], reg['y0'], reg['x1'], reg['y1']); i = inter(box, rb)
    return i / area(box) >= 0.10 or i / area(rb) >= 0.50

def fuse(yolo, pcr, names, tau):
    # yolo : [(cls, conf, x0, y0, x1, y1)] ; pcr : régions PatchCore (dict) ; sortie : régions au format du cahier des charges (§10.2)
    out = []
    for k, p in enumerate(sorted(yolo, key=lambda t: -t[1])):
        c, conf, x0, y0, x1, y1 = p; co = [r for r in pcr if overlaps((x0, y0, x1, y1), r)]
        out.append(dict(region_id=f'Y{k + 1}', source_engine='yolo', bbox=[round(x0, 1), round(y0, 1), round(x1 - x0, 1), round(y1 - y0, 1)], area=round((x1 - x0) * (y1 - y0), 1),
                        score=round(conf, 4), score_type='confiance du modèle (non calibrée, pas une probabilité)', classification=names[c],
                        classification_method='supervisé (YOLO)', label=f'Défaut proposé : {names[c]}',
                        corroboration=dict(engine='patchcore', score=round(max(r['score_max'] for r in co), 3), threshold=round(tau, 3)) if co else None, review_status='pending'))
    only = [r for r in pcr if not any(overlaps(p[2:], r) for p in yolo)]
    for k, r in enumerate(sorted(only, key=lambda r: -r['score_max'])):
        out.append(dict(region_id=f'P{k + 1}', source_engine='patchcore', bbox=[round(r['x0'], 1), round(r['y0'], 1), round(r['x1'] - r['x0'], 1), round(r['y1'] - r['y0'], 1)],
                        area=round((r['x1'] - r['x0']) * (r['y1'] - r['y0']), 1), score=round(r['score_max'], 3), score_type='distance L2 (brut, seuil %.3f)' % tau,
                        classification=None, classification_method=None, label='Anomalie suspectée — score de distance', corroboration=None, review_status='pending'))
    return out

def eval_image(gt, yolo, pcr):
    # gt : [(cls, x0, y0, x1, y1)]
    ys = sorted(yolo, key=lambda t: -t[1]); used = set(); tp = []
    for p in ys:
        best, bj = 0., -1
        for j, g in enumerate(gt):
            if j in used or g[0] != p[0]: continue
            v = iou(p[2:], g[1:])
            if v >= 0.5 and v > best: best, bj = v, j
        if bj >= 0: used.add(bj)
        tp.append(bj >= 0)
    pb = [(r['x0'], r['y0'], r['x1'], r['y1']) for r in pcr]
    return dict(gt_cls=[g[0] for g in gt], found_y=[j in used for j in range(len(gt))],
                found_p_loose=[any(inter(g[1:], b) / area(g[1:]) >= 0.10 for b in pb) for g in gt],
                found_p_strict=[any(iou(g[1:], b) >= 0.30 for b in pb) for g in gt],
                tp=tp, corro=[any(overlaps(p[2:], r) for r in pcr) for p in ys],
                pc_only=sum(1 for r in pcr if not any(overlaps(p[2:], r) for p in yolo)), n_yolo=len(yolo), n_pc=len(pcr))

def boot(found, total, n=2000, seed=0):
    found, total = np.asarray(found, float), np.asarray(total, float)
    if total.sum() == 0: return (float('nan'),) * 3
    idx = np.random.RandomState(seed).randint(0, len(found), (n, len(found))); r = found[idx].sum(1) / np.maximum(total[idx].sum(1), 1)
    return float(found.sum() / total.sum()), float(np.percentile(r, 2.5)), float(np.percentile(r, 97.5))

def aggregate(recs, names):
    n_gt = np.array([len(r['gt_cls']) for r in recs]); s = lambda k: np.array([sum(r[k]) for r in recs])
    fy, fl, fs = s('found_y'), s('found_p_loose'), s('found_p_strict')
    un_l = np.array([sum(a or b for a, b in zip(r['found_y'], r['found_p_loose'])) for r in recs])
    un_s = np.array([sum(a or b for a, b in zip(r['found_y'], r['found_p_strict'])) for r in recs])
    rows = {}
    for nm, f in (('YOLO seul', fy), ('PatchCore seul (recouvrement >= 10 % de la boîte)', fl), ('PatchCore seul (IoU >= 0.3)', fs),
                  ('Union YOLO + PatchCore (souple)', un_l), ('Union YOLO + PatchCore (IoU >= 0.3)', un_s)):
        r = boot(f, n_gt); rows[nm] = dict(rappel=round(r[0], 3), IC95=[round(r[1], 3), round(r[2], 3)])
    by = collections.defaultdict(lambda: [0, 0, 0, 0])            # n, YOLO, rattrapés (souple), rattrapés (strict)
    for r in recs:
        for c, fy_, fl_, fs_ in zip(r['gt_cls'], r['found_y'], r['found_p_loose'], r['found_p_strict']):
            b = by[names[c]]; b[0] += 1; b[1] += fy_; b[2] += (not fy_) and fl_; b[3] += (not fy_) and fs_
    tpf = [c for r in recs for t, c in zip(r['tp'], r['corro']) if t]; fpf = [c for r in recs for t, c in zip(r['tp'], r['corro']) if not t]
    nimg = len(recs)
    return dict(n_images=nimg, n_gt=int(n_gt.sum()), recalls=rows,
                rescued_by_patchcore_loose=int(sum(v[2] for v in by.values())), rescued_by_patchcore_strict=int(sum(v[3] for v in by.values())), missed_by_yolo=int(n_gt.sum() - fy.sum()),
                by_class={k: dict(n=v[0], yolo=v[1], rescued_loose=int(v[2]), rescued_strict=int(v[3])) for k, v in sorted(by.items())},
                yolo_fp_per_image=round(float(sum(sum(not t for t in r['tp']) for r in recs) / nimg), 3),
                patchcore_only_regions_per_image=round(float(np.mean([r['pc_only'] for r in recs])), 3),
                total_regions_per_image=round(float(np.mean([r['n_yolo'] + r['pc_only'] for r in recs])), 3),
                corroboration=dict(rate_among_yolo_true_positives=round(float(np.mean(tpf)), 3) if tpf else None, n_tp=len(tpf),
                                   rate_among_yolo_false_positives=round(float(np.mean(fpf)), 3) if fpf else None, n_fp=len(fpf)))

In [ ]:
# 5) Analyse d'une image : YOLO + PatchCore en coordonnées natives
def analyze(im):
    W, H = im.size
    r = model.predict(source=im, conf=CONF, imgsz=IMGSZ, iou=0.6, verbose=False)[0]; b = r.boxes
    yolo = [(int(c), float(s), *map(float, xy)) for c, s, xy in zip(b.cls.cpu().numpy(), b.conf.cpu().numpy(), b.xyxy.cpu().numpy())]
    pcr = rescale(regions_from_map(score_map(to_tensor(im)), TAU, MIN_AREA), 1 / SCALE)
    return yolo, pcr

def read_gt(split, p, W, H):
    out = []
    for line in (V2 / 'labels' / split / (Path(p).stem + '.txt')).read_text().splitlines():
        s = line.split()
        if len(s) == 5:
            c = int(s[0]); cx, cy, bw, bh = map(float, s[1:]); out.append((c, (cx-bw/2)*W, (cy-bh/2)*H, (cx+bw/2)*W, (cy+bh/2)*H))
    return out

def collect(split):
    recs = []
    for p in tqdm(sorted((V2 / 'images' / split).iterdir()), desc=split):
        im = ImageOps.exif_transpose(Image.open(p)).convert('RGB'); yolo, pcr = analyze(im)
        recs.append(eval_image(read_gt(split, p, *im.size), yolo, pcr))
    return recs

val_recs = collect('val'); VAL = aggregate(val_recs, NAMES)
print(json.dumps(VAL, indent=1, ensure_ascii=False))

In [ ]:
# 6) TEST : une seule fois (aucun paramètre n'a été réglé ; seuils YOLO et PatchCore hérités de la validation)
tfile = OUT / 'test_metrics.json'
if tfile.exists():
    print('Test déjà évalué : résultats rechargés.'); TEST = json.load(open(tfile))
else:
    TEST = aggregate(collect('test'), NAMES); json.dump(TEST, open(tfile, 'w'), indent=1, ensure_ascii=False)
print(json.dumps(TEST, indent=1, ensure_ascii=False))

In [ ]:
# 7) Tissus externes inconnus (optionnel) : alertes par image
EXT = {}
def run_external(folder):
    res = []
    for p in tqdm(sorted(f for f in folder.iterdir() if f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp', '.heic'}), desc=folder.name):
        im = ImageOps.exif_transpose(Image.open(p)).convert('RGB'); f = EXT_LONG_SIDE / max(im.size)
        if f < 1: im = im.resize((round(im.size[0] * f), round(im.size[1] * f)), Image.LANCZOS)
        yolo, pcr = analyze(im); fz = fuse(yolo, pcr, NAMES, TAU)
        res.append(dict(name=p.name, n_yolo=len(yolo), n_pc_only=sum(r['source_engine'] == 'patchcore' for r in fz), yolo_classes=[NAMES[y[0]] for y in yolo]))
    return res
for kind in ('normal', 'defect'):
    d = EXT_DIR / kind
    if d.exists() and any(d.iterdir()):
        r = run_external(d)
        EXT[kind] = dict(n_images=len(r), images_with_yolo=sum(x['n_yolo'] > 0 for x in r), images_with_patchcore_only=sum(x['n_pc_only'] > 0 for x in r),
                         images_with_any=sum((x['n_yolo'] + x['n_pc_only']) > 0 for x in r), mean_regions=round(float(np.mean([x['n_yolo'] + x['n_pc_only'] for x in r])), 2),
                         yolo_classes=dict(collections.Counter(c for x in r for c in x['yolo_classes'])))
        print(kind, EXT[kind])
if not EXT: print('Pas de dossier external_test/{normal,defect} : étape ignorée.')
else: print('NB : sur les images "normal", toute alerte est une fausse alerte (échelle de prise de vue supposée proche de celle du dataset).')

In [ ]:
# 8) Visualisation de la sortie fusionnée (6 images de validation) + exemple de JSON
import matplotlib.pyplot as plt
vimgs = sorted((V2 / 'images' / 'val').iterdir()); sample = random.Random(SEED).sample(vimgs, 6)
fig, axes = plt.subplots(3, 2, figsize=(14, 15)); example = None
for ax, p in zip(axes.flat, sample):
    im = ImageOps.exif_transpose(Image.open(p)).convert('RGB'); yolo, pcr = analyze(im); fz = fuse(yolo, pcr, NAMES, TAU); dr = ImageDraw.Draw(im); W = im.size[0]
    for c, x0, y0, x1, y1 in read_gt('val', p, *im.size): dr.rectangle([x0, y0, x1, y1], outline=(0, 255, 0), width=max(2, W // 400))
    for r in fz:
        x, y, w, h = r['bbox']; col = (255, 40, 40) if r['source_engine'] == 'yolo' else (255, 255, 255)
        dr.rectangle([x, y, x + w, y + h], outline=col, width=max(3, W // 250)); dr.text((x + 4, y + 4), r['label'] + (' [+PatchCore]' if r['corroboration'] else ''), fill=col)
        example = example or r
    ax.imshow(im); ax.set_title(f'{p.name[:30]} | vert = annotation, rouge = YOLO, blanc = PatchCore seul', fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()
print(json.dumps(example, indent=1, ensure_ascii=False))

In [ ]:
# 9) Export de la spécification de fusion + rapport à coller dans le chat
spec = dict(fusion_id='fusion_v1', rule=['toutes les détections YOLO (conf >= seuil) conservées, classe et score intacts',
                                         'région PatchCore recouvrant une détection YOLO : rattachée en corroboration, aucune modification',
                                         'région PatchCore sans équivalent YOLO : région distincte "Anomalie suspectée — score de distance", sans classe'],
            overlap_rule='>= 10 % de la boîte ou >= 50 % de la région', free_parameters='aucun (seuils hérités : YOLO conf=%.3f, PatchCore tau=%.3f)' % (CONF, TAU),
            yolo=dict(experiment=ycard['selected_experiment'], weights_sha256=ycard['weights_sha256'], conf=CONF, imgsz=IMGSZ),
            patchcore=dict(memory_sha256=json.load(open(PC_DIR / 'model_card.json'))['memory_sha256'], tau=TAU, min_region_area=MIN_AREA, scale=SCALE),
            dataset_hash=h, validation=VAL, test=TEST, external=EXT,
            limits=['PatchCore : critère de localisation plus souple que celui de YOLO (recouvrement >= 10 % de la boîte) ; version stricte IoU >= 0.3 également rapportée',
                    'régions PatchCore hors annotations = fausses alertes OU défauts non annotés (indistinguables ici)',
                    'la corroboration aide à prioriser la revue humaine ; elle ne remplace pas la décision de l opérateur',
                    'résultats valables pour des tissus proches de ceux du dataset (voir diagnostic de biais de domaine)'],
            created=datetime.datetime.now().isoformat(timespec='seconds'))
(OUT / 'fusion_spec.json').write_text(json.dumps(spec, indent=1, ensure_ascii=False, default=str), encoding='utf-8')
print(json.dumps(dict(validation=VAL, test=TEST, external=EXT), indent=1, ensure_ascii=False, default=str)[:11000])